# Neural Network Pricing of the Implied Volatility Surface

Ablation ladder (rungs 0–5) isolating model-class and feature-set effects. See report Methodology for the full rung-by-rung rationale and Results for interpretation.

| Rung | Model | Features |
|---|---|---|
| 0 | Constant | none |
| 1 | Polynomial deg 5 | logMoneyness, years |
| 2 | NN | logMoneyness, years |
| 3 | Polynomial deg 3 | + atmIV, realizedVol, recentReturn |
| 4 | NN | + atmIV, realizedVol, recentReturn |
| 5 | SVI (in-sample) | per-slice |

In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from nn_utils import (
    set_seed, train_model, predict, evaluate, per_ticker, bucket_errors,
)

pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:9.4f}")
set_seed(42)

print("torch", torch.__version__, "| threads", torch.get_num_threads())

df = pd.read_parquet("../data/processed/iv_surface.parquet")
print(f"{len(df):,} rows")

train_df = df[df.split == "train"].copy()
val_df   = df[df.split == "val"].copy()
test_t   = df[df.split == "test_temporal"].copy()
test_k   = df[df.split == "test_ticker"].copy()

TRAIN_MEAN = train_df.normalizedIV.mean()
print(f"train mean normalizedIV = {TRAIN_MEAN:.4f}")
print(df.groupby("split").size().to_string())

torch 2.13.0 | threads 8


4,445,206 rows


train mean normalizedIV = 1.3232
split
test_temporal     718467
test_ticker       276456
train            2651495
val               798788


In [2]:
# Stale-parquet guard. Parquets are shared through Drive and local copies go
# stale; build_dataset.py's build-time check cannot catch that.
assert (df.recentReturn.abs() > 1.0).sum() == 0, "stale parquet - rebuild it"
assert (df.midIV >= 9.99).sum() == 0, "stale parquet - sentinel midIV values present, rebuild it"


## 1. Benchmarks

Polynomial baseline (R²=0.161) and the absolute R² ceiling for models restricted to (logMoneyness, years), estimated via within-bin variance at increasing grid resolution. See report Methodology.

In [3]:
def ceiling(d, mstep, tbins, label=""):
    d = d.copy()
    d["mBin"] = pd.cut(d.logMoneyness, np.arange(-1.5, 1.5 + mstep, mstep))
    d["tBin"] = pd.cut(d.years, tbins)
    g = d.groupby(["mBin", "tBin"], observed=True).normalizedIV
    w, c = g.var(), g.size()
    ok = w.notna() & (c >= 30)
    irr = np.average(w[ok], weights=c[ok])
    return 1 - irr / d.normalizedIV.var(), 100 * c[ok].sum() / len(d)

GRIDS = [
    (0.10, [0, .05, .1, .25, .5, 1, 2, 5]),
    (0.05, [0, .02, .05, .1, .25, .5, 1, 2, 5]),
    (0.02, [0, .02, .04, .08, .12, .17, .25, .4, .6, 1, 1.5, 2, 3, 5]),
    (0.01, [0, .02, .04, .08, .12, .17, .25, .4, .6, 1, 1.5, 2, 3, 5]),
]

print(f"{'m step':>8} {'ceiling':>9} {'coverage':>10}")
for ms, tb in GRIDS:
    c, cov = ceiling(test_t, ms, tb)
    print(f"{ms:8.2f} {c:9.4f} {cov:9.1f}%")

CEILING, _ = ceiling(test_t, 0.01, GRIDS[-1][1])
print(f"\nCEILING (test_temporal) ~ {CEILING:.4f}")
print(f"polynomial baseline      = 0.1610")

  m step   ceiling   coverage
    0.10    0.4490      98.5%
    0.05    0.4586      98.5%


    0.02    0.4694      98.2%


    0.01    0.4799      97.3%

CEILING (test_temporal) ~ 0.4799
polynomial baseline      = 0.1610


## 2. Rung 2 — NN, Same Inputs as the Polynomial

Features: logMoneyness, years. Architecture: 3 hidden layers (64, 32, 16), ReLU. Target: log(normalizedIV). See report Methodology for training details.

In [4]:
SAMPLE_FRAC = None      # set to 0.25 for a quick iteration pass
EPOCHS = 40

F_BASE = ["logMoneyness", "years"]

m2, s2, h2 = train_model(train_df, val_df, F_BASE,
                         epochs=EPOCHS, sample_frac=SAMPLE_FRAC)

p2_val = predict(m2, s2, val_df,  F_BASE)
p2_t   = predict(m2, s2, test_t,  F_BASE)
p2_k   = predict(m2, s2, test_k,  F_BASE)

r2 = pd.DataFrame([
    evaluate(val_df, p2_val, "val",           TRAIN_MEAN),
    evaluate(test_t, p2_t,   "test_temporal", TRAIN_MEAN),
    evaluate(test_k, p2_k,   "test_ticker",   TRAIN_MEAN),
])
print("\nRUNG 2 - NN on (logMoneyness, years)")
print(r2.set_index("split"))
print(f"\n  polynomial deg5 a=2.89e5 : R2 0.1610, 11.74 vol pts (temporal)")
print(f"  ceiling for these inputs : {CEILING:.4f}")

  epoch   1  train 0.07328  val 0.05568  *


  epoch   2  train 0.06946  val 0.05496  *


  epoch   3  train 0.06909  val 0.05547


  epoch   4  train 0.06884  val 0.05518


  epoch   5  train 0.06868  val 0.05471  *


...

  epoch  10  train 0.06840  val 0.05451


  epoch  11  train 0.06833  val 0.05460


  epoch  12  train 0.06836  val 0.05458


  epoch  13  train 0.06833  val 0.05488
  early stop at epoch 13 (best val 0.05407)
  trained in 10.1s on 2,651,495 rows



RUNG 2 - NN on (logMoneyness, years)
                     R2  rmse_normIV  rmse_volpts  FLAT_volpts  vw_rmse_volpts  butterfly_%  calendar_%
split                                                                                                  
val              0.4415       0.4711      10.6985      13.9212          6.2220       8.6216      0.5284
test_temporal    0.3179       0.6213      11.5213      14.7823          6.1105       7.3385      0.6339
test_ticker      0.2594       0.3217      16.7045      17.3261         11.8858      10.6661      1.0245

  polynomial deg5 a=2.89e5 : R2 0.1610, 11.74 vol pts (temporal)
  ceiling for these inputs : 0.4799


## 3. Rung 3 — 5-Feature Polynomial Baseline

Loaded from `results/poly_test_results_5feat.csv` (computed in `polynomial.ipynb`), not retrained here.

In [5]:
poly_5feat_raw = pd.read_csv("../results/poly_test_results_5feat.csv", index_col=0)

def _split_key(raw_index):
    for name in ["test_temporal", "test_ticker"]:
        if raw_index.startswith(name):
            return name
    raise ValueError(raw_index)

POLY_5FEAT = {
    _split_key(idx): {"R2": row["r2"], "rmse_volpts": row["rmse_volpts"]}
    for idx, row in poly_5feat_raw.iterrows()
}
print("RUNG 3 - Polynomial (5 features), loaded from polynomial.ipynb")
print(pd.DataFrame(POLY_5FEAT).T)

RUNG 3 - Polynomial (5 features), loaded from polynomial.ipynb
                     R2  rmse_volpts
test_temporal    0.5074      10.2980
test_ticker     -0.0443      20.5748


## 4. Rung 4 — NN + Underlying Characteristics (Final Model)

Adds atmIV, realizedVol, recentReturn to logMoneyness and years. Same architecture as Rung 2. See report Methodology for feature rationale.

In [6]:
F_FULL = ["logMoneyness", "years", "atmIV", "realizedVol", "recentReturn"]

m3, s3, h3 = train_model(train_df, val_df, F_FULL,
                         epochs=EPOCHS, sample_frac=SAMPLE_FRAC)

p3_val = predict(m3, s3, val_df, F_FULL)
p3_t   = predict(m3, s3, test_t, F_FULL)
p3_k   = predict(m3, s3, test_k, F_FULL)

r3 = pd.DataFrame([
    evaluate(val_df, p3_val, "val",           TRAIN_MEAN),
    evaluate(test_t, p3_t,   "test_temporal", TRAIN_MEAN),
    evaluate(test_k, p3_k,   "test_ticker",   TRAIN_MEAN),
])
print("\nRUNG 4 - NN + underlying characteristics")
print(r3.set_index("split"))

  epoch   1  train 0.02739  val 0.01263  *


  epoch   2  train 0.00626  val 0.01108  *


  epoch   3  train 0.00509  val 0.01132


  epoch   4  train 0.00466  val 0.01062  *


  epoch   5  train 0.00440  val 0.01057  *


...

  epoch   9  train 0.00387  val 0.01009


  epoch  10  train 0.00377  val 0.01019


  epoch  11  train 0.00368  val 0.01001


  epoch  12  train 0.00359  val 0.01026
  early stop at epoch 12 (best val 0.00998)
  trained in 9.2s on 2,651,495 rows



RUNG 4 - NN + underlying characteristics
                     R2  rmse_normIV  rmse_volpts  FLAT_volpts  vw_rmse_volpts  butterfly_%  calendar_%
split                                                                                                  
val              0.9622       0.1226       3.7375      13.9212          2.5396       5.5403      0.1691
test_temporal    0.9639       0.1429       4.7230      14.7823          3.1764       5.8178      0.2182
test_ticker      0.7364       0.1920       8.1613      17.3261          6.3320       5.9304      0.7183


## 5. The Ladder

Full rung-by-rung comparison, both test sets. See report Results for interpretation.

In [7]:
POLY = {
    "test_temporal": {"R2": 0.1610, "rmse_volpts": 11.738},
    "test_ticker":   {"R2": 0.2288, "rmse_volpts": 15.030},
}
SVI = {
    "test_temporal": {"R2": float("nan"), "rmse_volpts": 4.67},
    "test_ticker":   {"R2": float("nan"), "rmse_volpts": 6.46},
}

ladder = []
for split in ["test_temporal", "test_ticker"]:
    flat = r2[r2.split == split].FLAT_volpts.iloc[0]
    ladder += [
        {"rung": "0  constant (train mean)",        "split": split,
         "R2": 0.0, "rmse_volpts": flat},
        {"rung": "1  poly deg5 (2-feat)",           "split": split, **POLY[split]},
        {"rung": "2  NN (2-feat)",                  "split": split,
         **r2[r2.split == split][["R2", "rmse_volpts"]].iloc[0].to_dict()},
        {"rung": "3  poly deg3 (5-feat)",           "split": split, **POLY_5FEAT[split]},
        {"rung": "4  NN (5-feat, final)",           "split": split,
         **r3[r3.split == split][["R2", "rmse_volpts"]].iloc[0].to_dict()},
        {"rung": "5  SVI (in-sample floor)",        "split": split, **SVI[split]},
    ]

L = pd.DataFrame(ladder).pivot(index="rung", columns="split",
                               values=["R2", "rmse_volpts"])
print(L)
print(f"\nceiling for f(logMoneyness, years) on test_temporal: {CEILING:.4f}")
print("\nRungs 0-2 are bounded by that ceiling. Rungs 3-4 are not - they use")
print("information that is not a function of the two contract coordinates.")
print("\nSVI (rung 5) is in-sample: it uses the day's own prices, so its RMSE")
print("is a fitting residual rather than a prediction error. Rung 4 narrows the")
print("gap to that floor but does not close it on either test set.")


                                    R2               rmse_volpts            
split                    test_temporal test_ticker test_temporal test_ticker
rung                                                                        
0  constant (train mean)        0.0000      0.0000       14.7823     17.3261
1  poly deg5 (2-feat)           0.1610      0.2288       11.7380     15.0300
2  NN (2-feat)                  0.3179      0.2594       11.5213     16.7045
3  poly deg3 (5-feat)           0.5074     -0.0443       10.2980     20.5748
4  NN (5-feat, final)           0.9639      0.7364        4.7230      8.1613
5  SVI (in-sample floor)           NaN         NaN        4.6700      6.4600

ceiling for f(logMoneyness, years) on test_temporal: 0.4799

Rungs 0-2 are bounded by that ceiling. Rungs 3-4 are not - they use
information that is not a function of the two contract coordinates.

SVI (rung 5) is in-sample: it uses the day's own prices, so its RMSE
is a fitting residual rather than a pre

In [8]:
from pathlib import Path

nn_figures_dir = Path("../results/figures/neural_network")
nn_figures_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (h, t) in zip(axes, [(h2, "rung 2  NN, 2-feat"),
                             (h3, "rung 4  NN, 5-feat (final)")]):
    ax.plot(h.epoch, h.train_loss, label="train")
    ax.plot(h.epoch, h.val_loss, label="val")
    ax.set_title(t); ax.set_xlabel("epoch"); ax.set_ylabel("MSE, log space")
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(nn_figures_dir / "nn_training_curves.png", dpi=150)
print(f"saved {nn_figures_dir / 'nn_training_curves.png'}")

saved ../results/figures/neural_network/nn_training_curves.png


## 6. Error Analysis

Per-ticker, per-bucket (maturity × moneyness), and vega-weighted error breakdowns for the final model (Rung 4). See report Results for interpretation.

In [9]:
best_p_t, best_p_k, best_label = p3_t, p3_k, "rung 4"

print(f"PER-TICKER, test_temporal ({best_label})")
print(per_ticker(test_t, best_p_t))
print(f"\nPER-TICKER, test_ticker ({best_label})")
print(per_ticker(test_k, best_p_k))
print("\nTSLA is 47% of test_ticker by row count. The aggregate number is")
print("largely a TSLA number - report per-ticker alongside it.")

PER-TICKER, test_temporal (rung 4)
                 n   share_%     atmIV        R2  rmse_normIV  rmse_volpts
ticker                                                                    
AAPL    50097.0000    6.9728    0.1891    0.9660       0.1434       2.6573
BA      39793.0000    5.5386    0.3518    0.7938       0.0749       2.6206
COST    64449.0000    8.9703    0.2299    0.9031       0.1145       2.5674
EOG     16785.0000    2.3362    0.2342    0.7418       0.0895       2.1899
GS      54351.0000    7.5649    0.2482    0.9258       0.0979       2.4618
LLY     71494.0000    9.9509    0.3323    0.7384       0.0981       3.3052
NVDA   179288.0000   24.9542    0.4697    0.8301       0.1866       8.4965
SPY    242210.0000   33.7121    0.1244    0.9804       0.1436       1.8633

PER-TICKER, test_ticker (rung 4)
                 n   share_%     atmIV        R2  rmse_normIV  rmse_volpts
ticker                                                                    
JPM     38063.0000   13.7682   

In [10]:
from pathlib import Path

fig_dir_summary = Path("../results/figures/results_summary")
fig_dir_summary.mkdir(parents=True, exist_ok=True)


def _fmt_per_ticker(df):
    d = df.copy()
    d["n"] = d["n"].map(lambda v: f"{int(v):,}")
    d["share_%"] = d["share_%"].map(lambda v: f"{v:.1f}%")
    d["atmIV"] = d["atmIV"].map(lambda v: f"{v:.3f}")
    d["R2"] = d["R2"].map(lambda v: f"{v:.3f}")
    d["rmse_normIV"] = d["rmse_normIV"].map(lambda v: f"{v:.3f}")
    d["rmse_volpts"] = d["rmse_volpts"].map(lambda v: f"{v:.2f}")
    return d.reset_index().rename(columns={
        "ticker": "Ticker", "n": "N", "share_%": "Share",
        "atmIV": "ATM IV", "R2": "R²",
        "rmse_normIV": "RMSE (normIV)", "rmse_volpts": "RMSE (vol pts)",
    })


def _draw_ticker_table(ax, data, title):
    col_labels = list(data.columns)
    cell_text = data.values.tolist()
    tbl = ax.table(cellText=cell_text, colLabels=col_labels, cellLoc="center", loc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1, 1.7)
    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor("#dddddd")
        cell.PAD = 0.04
        if row == 0:
            cell.set_facecolor("#2c3e50")
            cell.set_text_props(color="white", weight="bold")
        elif row % 2 == 0:
            cell.set_facecolor("#f7f7f7")
        if col == 0:
            cell.set_text_props(ha="left")
    ax.axis("off")
    ax.set_title(title, fontsize=11, fontweight="bold", pad=10)


pt_t = _fmt_per_ticker(per_ticker(test_t, best_p_t))
pt_k = _fmt_per_ticker(per_ticker(test_k, best_p_k))

fig, axes = plt.subplots(2, 1, figsize=(9, 2.6 + 0.42 * (len(pt_t) + len(pt_k))))
_draw_ticker_table(axes[0], pt_t, "test_temporal  (trained tickers, Nov–Dec)")
_draw_ticker_table(axes[1], pt_k, "test_ticker  (held-out tickers, Nov–Dec)")
fig.suptitle(f"{best_label}: Per-Ticker Performance", fontsize=13, fontweight="bold", y=0.99)
plt.tight_layout(rect=[0, 0, 1, 0.95])

out = fig_dir_summary / "per_ticker_table.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {out}")

Saved to ../results/figures/results_summary/per_ticker_table.png


/var/folders/5s/fj7mpgnn40b92nyxpk35ysw40000gn/T/ipykernel_69098/2016407452.py:54: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
print(f"RMSE (vol pts) by maturity x moneyness - test_temporal, {best_label}")
print(bucket_errors(test_t, best_p_t))
print(f"\nsame, test_ticker")
print(bucket_errors(test_k, best_p_k))

RMSE (vol pts) by maturity x moneyness - test_temporal, rung 4
mBucket  deep OTM put   OTM put  near ATM  OTM call  deep OTM call
tBucket                                                           
<1m           12.5472    4.8748    3.2821    5.4351         6.3922
1-3m           8.0692    2.7420    1.9149    2.8886         3.9259
3-6m           5.9824    2.6862    2.7760    4.4058         4.3278
>6m            5.7579    3.4700    3.5801    5.1513         6.7790

same, test_ticker
mBucket  deep OTM put   OTM put  near ATM  OTM call  deep OTM call
tBucket                                                           
<1m           20.4482    9.5937    7.4537    9.0677        21.8870
1-3m          13.8297    6.1009    5.2863    6.5242        18.8216
3-6m           8.1620    4.4641    4.6391    5.5357         8.4137
>6m            6.3163    3.9211    4.2224    4.9413         7.8616


In [12]:
tb_order = ["<1m", "1-3m", "3-6m", ">6m"]
mb_order = ["deep OTM put", "OTM put", "near ATM", "OTM call", "deep OTM call"]

be_t = bucket_errors(test_t, best_p_t).reindex(index=tb_order, columns=mb_order)
be_k = bucket_errors(test_k, best_p_k).reindex(index=tb_order, columns=mb_order)
vmax = max(be_t.values.max(), be_k.values.max())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, data, title in [
    (axes[0], be_t, "test_temporal  (trained tickers)"),
    (axes[1], be_k, "test_ticker  (held-out tickers)"),
]:
    im = ax.imshow(data.values, cmap="YlOrRd", vmin=0, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(mb_order)))
    ax.set_xticklabels(mb_order, rotation=30, ha="right", fontsize=8.5)
    ax.set_yticks(range(len(tb_order)))
    ax.set_yticklabels(tb_order, fontsize=9)
    ax.set_title(title, fontsize=10.5)
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data.values[i, j]
            color = "white" if v > vmax * 0.6 else "black"
            ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=9, color=color)

cbar = fig.colorbar(im, ax=axes, shrink=0.85, pad=0.02)
cbar.set_label("RMSE (vol pts)")
fig.suptitle(f"{best_label}: Error by Maturity × Moneyness Bucket", fontsize=12, y=1.03)

out = fig_dir_summary / "error_heatmap.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {out}")

Saved to ../results/figures/results_summary/error_heatmap.png


/var/folders/5s/fj7mpgnn40b92nyxpk35ysw40000gn/T/ipykernel_69098/3187256818.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
# Where does the error actually live, and does any of it matter economically?
for label, d, p in [("test_temporal", test_t, best_p_t),
                    ("test_ticker",   test_k, best_p_k)]:
    err_norm = d.normalizedIV.to_numpy() - p
    err_vol = err_norm * d.atmIV.to_numpy()
    rmse = 100 * np.sqrt((err_vol ** 2).mean())
    print(f"\n{label}")
    print(f"  vol-pt RMSE, unweighted   : {rmse:7.2f}")
    if "ve" in d.columns:
        vw = 100 * np.sqrt(np.average(err_vol ** 2, weights=d.ve.abs()))
        print(f"  vol-pt RMSE, vega-weighted: {vw:7.2f}   <- the error that costs money")
    tail = d.normalizedIV > 5.0
    if tail.any():
        share = 100 * (err_vol[tail.to_numpy()] ** 2).sum() / (err_vol ** 2).sum()
        print(f"  rows with normalizedIV > 5: {tail.sum():,} ({100*tail.mean():.3f}%)"
              f" carrying {share:.1f}% of MSE")
    print(f"  corr(|err|, atmIV)        : "
          f"{np.corrcoef(np.abs(err_norm), d.atmIV)[0,1]:+7.3f}")


test_temporal
  vol-pt RMSE, unweighted   :    4.72
  vol-pt RMSE, vega-weighted:    3.18   <- the error that costs money
  rows with normalizedIV > 5: 5,470 (0.761%) carrying 1.1% of MSE
  corr(|err|, atmIV)        :  +0.135

test_ticker
  vol-pt RMSE, unweighted   :    8.16
  vol-pt RMSE, vega-weighted:    6.33   <- the error that costs money
  rows with normalizedIV > 5: 25 (0.009%) carrying 0.3% of MSE
  corr(|err|, atmIV)        :  +0.149


## 7. Saving Results

In [14]:
out = pd.concat([
    r2.assign(model="nn_raw"),
    r3.assign(model="nn_full"),
])[["model", "split", "R2", "rmse_normIV", "rmse_volpts", "FLAT_volpts",
    "vw_rmse_volpts", "butterfly_%", "calendar_%"]]
out.to_csv("../results/nn_test_results.csv", index=False)
L.to_csv("../results/nn_ladder.csv")
print(out.to_string(index=False))
print("\nwrote ../results/nn_test_results.csv and ../results/nn_ladder.csv")

  model         split        R2  rmse_normIV  rmse_volpts  FLAT_volpts  vw_rmse_volpts  butterfly_%  calendar_%
 nn_raw           val    0.4415       0.4711      10.6985      13.9212          6.2220       8.6216      0.5284
 nn_raw test_temporal    0.3179       0.6213      11.5213      14.7823          6.1105       7.3385      0.6339
 nn_raw   test_ticker    0.2594       0.3217      16.7045      17.3261         11.8858      10.6661      1.0245
nn_full           val    0.9622       0.1226       3.7375      13.9212          2.5396       5.5403      0.1691
nn_full test_temporal    0.9639       0.1429       4.7230      14.7823          3.1764       5.8178      0.2182
nn_full   test_ticker    0.7364       0.1920       8.1613      17.3261          6.3320       5.9304      0.7183

wrote ../results/nn_test_results.csv and ../results/nn_ladder.csv
